# Обучение голоса в Kaggle Notebooks

Бесплатный Kaggle: P100 (16 ГБ) либо T4 x2, 30 часов GPU в неделю (сброс в субботу
в 00:00 UTC), одна GPU-сессия — до 9 часов.

Два обязательных переключателя в панели справа: **Accelerator → GPU** и
**Internet → On** (интернет доступен только на аккаунте с подтверждённым телефоном;
без него не установятся пакеты и не скачаются предобученные веса).

`/kaggle/working` живёт только внутри сессии — сохраняйте результат через
**Save Version → Save & Run All** или скачивайте архив.


## 1. Проверить GPU и интернет


In [ ]:
!nvidia-smi
import torch, socket
assert torch.cuda.is_available(), 'включите Accelerator -> GPU в панели справа'
try:
    socket.create_connection(('pypi.org', 443), timeout=5).close()
except OSError:
    raise SystemExit('включите Internet -> On в панели справа')
print('GPU:', torch.cuda.get_device_name(0))


## 2. Установка

Applio ставит свои версии torch и transformers поверх образа Kaggle — это несколько
минут и несколько гигабайт загрузки.


In [ ]:
import os, pathlib

VC = 'rvc'   # 'rvc' (Applio) или 'sovits' (so-vits-svc-fork)
WORK = pathlib.Path('/kaggle/working')
for name in ('profiles', 'voices', 'logs'):
    (WORK / name).mkdir(exist_ok=True)

!git clone --depth 1 -b claude/voice-cloning-text-synthesis-7mnlwx https://github.com/tyetladd/RustTraining /kaggle/working/RustTraining 2>/dev/null || true
%pip install -q -e '/kaggle/working/RustTraining/voice-clone-tts[stress,asr,silero]'

if VC == 'rvc':
    !git clone --depth 1 https://github.com/IAHispano/Applio /kaggle/working/Applio 2>/dev/null || true
    %pip install -q -r /kaggle/working/Applio/requirements.txt
    os.environ['VCTTS_APPLIO_DIR'] = '/kaggle/working/Applio'
else:
    %pip install -q -U so-vits-svc-fork

!vctts converters


## 3. Запись голоса

Загрузите записи как Kaggle Dataset (**+ Add Data → Upload**): он монтируется
в `/kaggle/input/<датасет>/` и переживает перезапуск сессии, в отличие от
`/kaggle/working`.


In [ ]:
SPEAKER = 'anna'
candidates = sorted(pathlib.Path('/kaggle/input').rglob('*.mp3'))
candidates += sorted(pathlib.Path('/kaggle/input').rglob('*.wav'))
assert candidates, 'добавьте датасет с записью через + Add Data'
SOURCE = candidates[0]
print('запись:', SOURCE)


## 4. Профиль диктора


In [ ]:
cmd = f'vctts profile build "{SOURCE}" -o "{WORK}/profiles/{SPEAKER}" --name {SPEAKER} --overwrite'
!{cmd}


## 5. Обучение

Сессия обрывается на 9 часах, поэтому режьте обучение на части: запускайте
посильное число эпох, сохраняйте версию, продолжайте в следующей сессии.
Чекпоинты пишутся в `/kaggle/working/logs` и попадают в output сохранённой версии.


In [ ]:
EPOCHS = 150
BATCH_SIZE = 8
SAVE_EVERY = 25

def train_command(resume=False, epochs=None):
    """Собрать команду обучения: у драйверов разные полезные опции."""
    parts = [
        'vctts voice train',
        f'-p "{WORK}/profiles/{SPEAKER}"',
        f'-o "{WORK}/voices/{SPEAKER}"',
        f'--vc {VC}',
        f'--epochs {epochs or EPOCHS}',
        '--resume' if resume else '--overwrite',
        f'--vc-option batch_size={BATCH_SIZE}',
    ]
    if VC == 'rvc':
        # Чекпоинты Applio должны лежать вне эфемерного чекаута.
        parts += [f'--vc-option logs_dir="{WORK}/logs"',
                  f'--vc-option save_every_epoch={SAVE_EVERY}']
    else:
        # so-vits-svc хранит всё внутри -o, эта папка уже персистентная.
        parts += ['--sample-rate 44100']
    return ' '.join(parts)

print(train_command())


In [ ]:
cmd = train_command()
!{cmd}


### Продолжить в новой сессии

Добавьте output прошлой версии как входные данные (**+ Add Data → Your Work**),
скопируйте `logs/` и `voices/` обратно в `/kaggle/working` и запустите продолжение.


In [ ]:
# import shutil
# previous = pathlib.Path('/kaggle/input/<ваша-прошлая-версия>')
# shutil.copytree(previous / 'logs', WORK / 'logs', dirs_exist_ok=True)
# shutil.copytree(previous / 'voices', WORK / 'voices', dirs_exist_ok=True)

cmd = train_command(resume=True)
!{cmd}


## 6. Проверка


In [ ]:
TEXT = 'Проверка синтеза. Старинный замок на горе, а на двери замок.'

cmd = (f'vctts speak -b silero -l ru --voice-model "{WORK}/voices/{SPEAKER}" '
       f'--transpose auto -t "{TEXT}" -o "{WORK}/sample.wav"')
!{cmd}

from IPython.display import Audio
Audio(str(WORK / 'sample.wav'))


## 7. Забрать модель

Архив появится в output ноутбука — скачайте его из интерфейса Kaggle после
**Save Version**.


In [ ]:
import shutil
archive = shutil.make_archive(str(WORK / f'{SPEAKER}-voice'), 'zip',
                              WORK / 'voices' / SPEAKER)
print('готово:', archive)
